In [ ]:
import kaggle_benchmarks as kbench
import json
import re
import math
from datetime import datetime

# ----------------------------
# Global trace store
# ----------------------------
TRACE_LOG = []

FAILURE_MODES = [
    "failure_to_recognize_key_aspects",
    "hallucination",
    "misapplication_of_equation_or_model",
    "incorrect_factual_knowledge",
    "calculation_error",
]

# ----------------------------
# Helpers
# ----------------------------
def extract_json(text):
    if not text: return None
    fence = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if fence: blob = fence.group(1)
    else:
        start, end = text.find("{"), text.rfind("}")
        if start == -1 or end == -1 or end <= start: return None
        blob = text[start:end+1]
    try: return json.loads(blob)
    except: return None

def numeric_pass(answer_text, target, rel_tol=0.015):
    def parse_val(t):
        s = str(t).replace(" ", "").replace(",", "").lower()
        s = re.sub(r"\\times10\^?{?(-?\d+)}?", r"e\1", s)
        m = re.search(r"[-+]?\d*\.?\d+(?:[eE^][-+]?\d+)?", s)
        if m: 
            try: return float(m.group(0).replace("^", "e"))
            except: return None
        return None
    pred = parse_val(answer_text)
    if pred is None: return False
    try:
        t_val = float(target)
        if t_val == 0: return abs(pred) < 1e-9
        return math.isclose(pred, t_val, rel_tol=rel_tol)
    except: return False

def classify_failure_fp_14(answer_text):
    if not answer_text or len(str(answer_text)) < 2: return "hallucination"
    return "calculation_error" # Default for numerical tasks

def build_trace(*, task_id, llm, prompt, response, parsed, final_answer, passed, failure_mode):
    return {
        "timestamp_utc": datetime.utcnow().isoformat() + "Z",
        "task_id": task_id,
        "model": str(llm),
        "pass": bool(passed),
        "failure_mode": failure_mode,
        "final_answer": final_answer,
        "raw_output": response,
        "parsed_output": parsed,
        "prompt": prompt
    }

# ----------------------------
# PT-Symmetric Parity Im(P12)
# ----------------------------
@kbench.task(name="FP-14 PT-Symmetric Parity Im(P12)", description="Physics")
def fp_14_pt_symmetric_parity_im_p(llm) -> tuple[int, int]:
    prompt = r"""A two-mode gain/loss device evolves as i*d/dt(psi)=H*psi in the basis psi=(a_L,a_R)^T, with H=[[i*gamma, kappa*e^(i*theta)], [kappa*e^(-i*theta), -i*gamma]] where gamma>0, kappa>0, and theta in R. Time reversal is complex conjugation T(psi)=psi*, so PT symmetry means P H* P = H. In this phase-coupled system, T inverts the phase parameter theta. A valid parity operator P must satisfy P_dag=P and P^2=I. Impose the convention that P(theta) depends continuously on theta and satisfies P(0)=[[0,1],[1,0]]. Fix theta=pi/3. What is Im(P_12)? (Give answer as a decimal rounded to 3 places, i.e., sqrt(3)/2).

Return JSON only in the following format:
{
  "final_answer": "<numeric value>"
}"""
    response = llm.prompt(prompt)
    parsed = extract_json(response)
    passed_checks = 0
    final_answer = ""
    failure_mode = None

    if parsed is None:
        failure_mode = "hallucination"
    else:
        final_answer = parsed.get("final_answer", "")
        if numeric_pass(final_answer, 0.866):
            passed_checks = 1
        else:
            failure_mode = classify_failure_fp_14(final_answer)

    trace = build_trace(
        task_id="fp_14",
        llm=llm, prompt=prompt, response=response, parsed=parsed, 
        final_answer=final_answer, passed=(passed_checks == 1), failure_mode=failure_mode
    )
    TRACE_LOG.append(trace)
    return (passed_checks, 1)


In [ ]:
fp_14_pt_symmetric_parity_im_p.run(kbench.llm)


In [ ]:
results = fp_14_pt_symmetric_parity_im_p.evaluate(llm=[kbench.llm])
results.as_dataframe()

In [ ]:
import pandas as pd
trace_df = pd.DataFrame(TRACE_LOG)
trace_df

In [ ]:
trace_df["failure_mode"].value_counts(dropna=False)

In [ ]:
trace_df.to_csv("fp_14_trace_log.csv", index=False)